# ElementalTask-RML
## Notebook 04 — Compositional Forecasting

This notebook explores whether component-task emergence and function-vector geometry can forecast compositional capability emergence before convergence.

**Goal:**
- estimate composite-task emergence checkpoints,
- compare predicted vs observed emergence,
- measure forecasting error,
- evaluate lightweight trajectory prediction heuristics.

Emergence ≠ magic. Monitor constraints. 📐

## 1. Setup

This notebook is designed to run from either:

- the repository root, or
- `notebooks_rml/` in Colab/GitHub.

It uses outputs from Notebooks 01–03 when available, and falls back to a small deterministic example so the notebook remains runnable.

In [ ]:
from pathlib import Path
import math
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

warnings.filterwarnings("ignore")

# Locate repo / notebook directory.
cwd = Path.cwd()
if cwd.name == "notebooks_rml":
    NOTEBOOK_DIR = cwd
    REPO_ROOT = cwd.parent
elif (cwd / "notebooks_rml").exists():
    REPO_ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks_rml"
else:
    # Colab fallback: assume user uploaded/copied notebook into repo root or current folder.
    REPO_ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks_rml"

FIG_DIR = NOTEBOOK_DIR / "figures"
RESULTS_DIR = NOTEBOOK_DIR / "results"
DOCS_DIR = NOTEBOOK_DIR / "docs"

for d in [NOTEBOOK_DIR, FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("FIG_DIR:", FIG_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 2. Load previous notebook outputs

Expected inputs when available:

- `results/01_emergence_order_table.csv`
- `results/02_fv_similarity.csv`
- `results/02_fv_drift_scores.csv`
- `results/03_monitoring_flags.csv`

If these files are missing or have different column names, this notebook creates a deterministic fallback dataset matching the earlier figures.

In [ ]:
def read_csv_if_exists(path: Path):
    if path.exists():
        print(f"Loaded: {path}")
        return pd.read_csv(path)
    print(f"Missing: {path}")
    return None

emergence_path = RESULTS_DIR / "01_emergence_order_table.csv"
fv_similarity_path = RESULTS_DIR / "02_fv_similarity.csv"
fv_drift_path = RESULTS_DIR / "02_fv_drift_scores.csv"
flags_path = RESULTS_DIR / "03_monitoring_flags.csv"

emergence_df = read_csv_if_exists(emergence_path)
fv_similarity_df = read_csv_if_exists(fv_similarity_path)
fv_drift_df = read_csv_if_exists(fv_drift_path)
flags_df = read_csv_if_exists(flags_path)

print("\nShapes:")
for name, df in [("emergence_df", emergence_df), ("fv_similarity_df", fv_similarity_df), ("fv_drift_df", fv_drift_df), ("flags_df", flags_df)]:
    print(name, None if df is None else df.shape)

## 3. Normalize emergence table

We need a table with at least:

| task | emergence_checkpoint | task_family |
|---|---:|---|

The loader below tries several common column names and then falls back to deterministic example data.

In [ ]:
def normalize_emergence_table(df):
    if df is None or df.empty:
        return None

    cols = {c.lower(): c for c in df.columns}

    task_col = None
    for candidate in ["task", "task_name", "name"]:
        if candidate in cols:
            task_col = cols[candidate]
            break

    ckpt_col = None
    for candidate in ["emergence_checkpoint", "first_checkpoint", "checkpoint", "checkpoint_order", "emerged_at"]:
        if candidate in cols:
            ckpt_col = cols[candidate]
            break

    if task_col is None or ckpt_col is None:
        print("Could not infer task/checkpoint columns from emergence table:", df.columns.tolist())
        return None

    out = df[[task_col, ckpt_col]].rename(columns={task_col: "task", ckpt_col: "emergence_checkpoint"}).copy()
    out["emergence_checkpoint"] = pd.to_numeric(out["emergence_checkpoint"], errors="coerce")
    out = out.dropna(subset=["task", "emergence_checkpoint"])

    if "task_family" in df.columns:
        out["task_family"] = df["task_family"].values[:len(out)]
    else:
        out["task_family"] = out["task"].astype(str).apply(lambda x: x.split(":")[0] if ":" in x else "unknown")

    return out.sort_values("emergence_checkpoint").reset_index(drop=True)

emergence = normalize_emergence_table(emergence_df)

if emergence is None or emergence.empty:
    emergence = pd.DataFrame({
        "task": [
            "simple:copying",
            "simple:uppercase",
            "simple:first_letter",
            "math:arithmetic",
            "compositional:copy_then_uppercase",
            "compositional:first_letter_then_uppercase",
        ],
        "emergence_checkpoint": [3000, 5000, 20000, 20000, 100000, 100000],
        "task_family": ["simple", "simple", "simple", "math", "compositional", "compositional"],
    })
    print("Using deterministic fallback emergence table.")

emergence

## 4. Define composite relationships

These component → composite relationships are intentionally simple and readable.

They can be expanded later from ElementalTask metadata or task definitions.

In [ ]:
COMPOSITE_RELATIONSHIPS = [
    {
        "composite": "compositional:copy_then_uppercase",
        "components": ["simple:copying", "simple:uppercase"],
    },
    {
        "composite": "compositional:first_letter_then_uppercase",
        "components": ["simple:first_letter", "simple:uppercase"],
    },
]

# Keep only relationships with known tasks in the emergence table.
available_tasks = set(emergence["task"])
relationships = []
for rel in COMPOSITE_RELATIONSHIPS:
    if rel["composite"] in available_tasks and all(c in available_tasks for c in rel["components"]):
        relationships.append(rel)

relationships

## 5. Load or construct function-vector similarity

Notebook 04 uses FV geometry as a forecasting correction.

If an actual `02_fv_similarity.csv` exists, we use it. Otherwise, we construct a deterministic toy similarity matrix where each composite task is close to its components.

In [ ]:
def normalize_similarity_table(df, tasks):
    if df is None or df.empty:
        return None

    # Case 1: square matrix with first column as task names or index-like column.
    possible = df.copy()
    if possible.shape[1] >= 2:
        first_col = possible.columns[0]
        if possible[first_col].astype(str).isin(tasks).any():
            possible = possible.set_index(first_col)

    if set(tasks).issubset(set(possible.index.astype(str))) and set(tasks).issubset(set(map(str, possible.columns))):
        mat = possible.copy()
        mat.index = mat.index.astype(str)
        mat.columns = mat.columns.astype(str)
        mat = mat.loc[tasks, tasks]
        return mat.astype(float)

    # Case 2: long table with task_a, task_b, similarity.
    cols = {c.lower(): c for c in df.columns}
    if {"task_a", "task_b", "similarity"}.issubset(cols):
        out = pd.DataFrame(np.eye(len(tasks)), index=tasks, columns=tasks)
        for _, row in df.iterrows():
            a = str(row[cols["task_a"]])
            b = str(row[cols["task_b"]])
            s = float(row[cols["similarity"]])
            if a in out.index and b in out.columns:
                out.loc[a, b] = s
                out.loc[b, a] = s
        return out

    print("Could not infer FV similarity format from:", df.columns.tolist())
    return None

tasks = emergence["task"].tolist()
fv_sim = normalize_similarity_table(fv_similarity_df, tasks)

if fv_sim is None:
    rng = np.random.default_rng(42)
    base = rng.normal(0, 0.12, size=(len(tasks), len(tasks)))
    mat = (base + base.T) / 2
    np.fill_diagonal(mat, 1.0)
    fv_sim = pd.DataFrame(mat, index=tasks, columns=tasks)

    # Inject component-composite structure.
    for rel in relationships:
        comp = rel["composite"]
        for c in rel["components"]:
            fv_sim.loc[comp, c] = fv_sim.loc[c, comp] = 0.72
    print("Using deterministic fallback FV similarity matrix.")

fv_sim.round(3)

## 6. Forecast heuristic

For each composite task:

```text
predicted checkpoint = mean(component checkpoints) + geometry penalty + instability penalty
```

Interpretation:

- component emergence provides a baseline,
- stronger component-composite FV similarity reduces penalty,
- instability increases penalty.

This is not a learned model. It is a transparent monitoring heuristic.

In [ ]:
# Optional monitoring summary from Notebook 03.
def infer_global_instability(flags_df, fv_drift_df):
    # Prefer an explicit stability/status table if present.
    if flags_df is not None and not flags_df.empty:
        numeric_cols = [c for c in flags_df.columns if pd.api.types.is_numeric_dtype(flags_df[c])]
        if numeric_cols:
            vals = flags_df[numeric_cols].select_dtypes(include=[np.number]).to_numpy().ravel()
            vals = vals[np.isfinite(vals)]
            if len(vals):
                # Conservative: high instability if many monitoring scores are low.
                return float(np.clip(1.0 - np.nanmean(vals), 0.0, 1.0))

    if fv_drift_df is not None and not fv_drift_df.empty:
        numeric_cols = [c for c in fv_drift_df.columns if pd.api.types.is_numeric_dtype(fv_drift_df[c])]
        if numeric_cols:
            vals = fv_drift_df[numeric_cols].select_dtypes(include=[np.number]).to_numpy().ravel()
            vals = vals[np.isfinite(vals)]
            if len(vals):
                # Normalize rough drift to [0, 1].
                return float(np.clip(np.nanmean(np.abs(vals)), 0.0, 1.0))

    return 0.15

global_instability = infer_global_instability(flags_df, fv_drift_df)
print("Global instability penalty factor:", round(global_instability, 3))

ckpt_lookup = dict(zip(emergence["task"], emergence["emergence_checkpoint"]))
max_ckpt = float(emergence["emergence_checkpoint"].max())

forecast_rows = []
for rel in relationships:
    comp = rel["composite"]
    comps = rel["components"]
    component_ckpts = np.array([ckpt_lookup[c] for c in comps], dtype=float)
    observed = float(ckpt_lookup[comp])

    mean_component = float(component_ckpts.mean())
    max_component = float(component_ckpts.max())
    component_spread = float(component_ckpts.max() - component_ckpts.min())

    similarities = np.array([fv_sim.loc[comp, c] for c in comps], dtype=float)
    mean_similarity = float(np.nanmean(similarities))

    # Similarity in [-1, 1] mapped to geometry cost in [0, 1].
    geometry_cost = float(np.clip(1.0 - ((mean_similarity + 1.0) / 2.0), 0.0, 1.0))

    # Scale penalties to observed checkpoint range.
    geometry_penalty = geometry_cost * 0.35 * max_ckpt
    instability_penalty = global_instability * 0.15 * max_ckpt
    spread_penalty = (component_spread / max_ckpt) * 0.10 * max_ckpt

    predicted = max_component + geometry_penalty + instability_penalty + spread_penalty
    predicted = float(np.clip(predicted, 0, max_ckpt))

    error = predicted - observed
    abs_error = abs(error)
    status = "accurate" if abs_error <= 0.15 * max_ckpt else ("overpredicted" if error > 0 else "underpredicted")

    forecast_rows.append({
        "composite": comp,
        "components": " + ".join(comps),
        "mean_component_checkpoint": mean_component,
        "max_component_checkpoint": max_component,
        "observed_checkpoint": observed,
        "mean_fv_similarity_to_components": mean_similarity,
        "geometry_cost": geometry_cost,
        "global_instability": global_instability,
        "predicted_checkpoint": predicted,
        "signed_error": error,
        "absolute_error": abs_error,
        "status": status,
    })

forecast_df = pd.DataFrame(forecast_rows)
forecast_df

## 7. Forecast metrics

We report simple, interpretable errors:

- mean absolute error,
- mean signed error,
- normalized MAE.

With two composite tasks, these metrics are diagnostic only, not a benchmark.

In [ ]:
if not forecast_df.empty:
    mae = float(forecast_df["absolute_error"].mean())
    mse = float((forecast_df["signed_error"] ** 2).mean())
    rmse = math.sqrt(mse)
    mean_signed_error = float(forecast_df["signed_error"].mean())
    normalized_mae = mae / max_ckpt if max_ckpt else np.nan
else:
    mae = rmse = mean_signed_error = normalized_mae = np.nan

metrics = pd.DataFrame([{
    "n_composites": len(forecast_df),
    "mae": mae,
    "rmse": rmse,
    "mean_signed_error": mean_signed_error,
    "normalized_mae": normalized_mae,
}])
metrics

## 8. Figure — predicted vs observed composite emergence

In [ ]:
if forecast_df.empty:
    print("No composite relationships available for forecasting.")
else:
    x = np.arange(len(forecast_df))
    width = 0.35

    plt.figure(figsize=(12, 6))
    plt.bar(x - width/2, forecast_df["observed_checkpoint"], width, label="observed")
    plt.bar(x + width/2, forecast_df["predicted_checkpoint"], width, label="predicted")
    plt.xticks(x, forecast_df["composite"], rotation=20, ha="right")
    plt.ylabel("Emergence checkpoint")
    plt.title("ElementalTask-RML: Composite Forecast vs Observed")
    plt.legend()
    plt.tight_layout()
    path = FIG_DIR / "04_forecast_vs_actual.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

## 9. Forecast confidence

A transparent confidence score:

```text
confidence = FV similarity × stability × low-error proxy
```

For this first notebook, confidence is descriptive and conservative.

In [ ]:
if not forecast_df.empty:
    # Similarity mapped from [-1, 1] to [0, 1].
    sim_conf = (forecast_df["mean_fv_similarity_to_components"] + 1.0) / 2.0
    stability_conf = 1.0 - forecast_df["global_instability"].clip(0, 1)
    spread_conf = 1.0 - (forecast_df["absolute_error"] / max_ckpt).clip(0, 1)

    forecast_df["forecast_confidence"] = (sim_conf * stability_conf * spread_conf).clip(0, 1)
else:
    forecast_df["forecast_confidence"] = []

forecast_df[["composite", "forecast_confidence", "status"]] if not forecast_df.empty else forecast_df

In [ ]:
if not forecast_df.empty:
    plt.figure(figsize=(12, 5))
    plt.bar(forecast_df["composite"], forecast_df["forecast_confidence"])
    plt.ylim(0, 1.05)
    plt.ylabel("Forecast confidence")
    plt.title("ElementalTask-RML: Compositional Forecast Confidence")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    path = FIG_DIR / "04_forecast_confidence.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

## 10. Early forecast windows

This section asks:

> How early could a monitoring system begin forecasting composite emergence?

We simulate forecast availability at checkpoint cutoffs and report whether the necessary component evidence has appeared.

In [ ]:
cutoffs = sorted(set([5000, 10000, 20000, 50000, int(max_ckpt)]))
window_rows = []

for cutoff in cutoffs:
    for rel in relationships:
        comp = rel["composite"]
        comps = rel["components"]
        comp_ckpts = [ckpt_lookup[c] for c in comps]
        observed = ckpt_lookup[comp]

        components_seen = all(c <= cutoff for c in comp_ckpts)
        composite_seen = observed <= cutoff

        if components_seen:
            component_mean = float(np.mean(comp_ckpts))
            mean_similarity = float(np.mean([fv_sim.loc[comp, c] for c in comps]))
            rough_prediction = max(comp_ckpts) + (1 - ((mean_similarity + 1) / 2)) * 0.35 * max_ckpt
            rough_prediction = float(np.clip(rough_prediction, 0, max_ckpt))
            rough_error = rough_prediction - observed
            forecast_available = True
        else:
            rough_prediction = np.nan
            rough_error = np.nan
            forecast_available = False

        window_rows.append({
            "cutoff_checkpoint": cutoff,
            "composite": comp,
            "components_seen": components_seen,
            "composite_seen": composite_seen,
            "forecast_available": forecast_available,
            "window_prediction": rough_prediction,
            "observed_checkpoint": observed,
            "window_signed_error": rough_error,
        })

window_df = pd.DataFrame(window_rows)
window_df

In [ ]:
if not window_df.empty:
    plot_df = window_df[window_df["forecast_available"]].copy()
    if not plot_df.empty:
        plt.figure(figsize=(12, 6))
        for comp, sub in plot_df.groupby("composite"):
            plt.plot(sub["cutoff_checkpoint"], sub["window_prediction"], marker="o", label=f"predicted: {comp}")
            observed = sub["observed_checkpoint"].iloc[0]
            plt.axhline(observed, linestyle="--", alpha=0.5)
        plt.xlabel("Forecast cutoff checkpoint")
        plt.ylabel("Predicted composite emergence checkpoint")
        plt.title("ElementalTask-RML: Early Forecast Windows")
        plt.legend()
        plt.tight_layout()
        path = FIG_DIR / "04_early_forecast_windows.png"
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved:", path)
    else:
        print("No early forecast windows available: components not observed before cutoffs.")

## 11. Save results

Outputs:

- `results/04_compositional_forecasts.csv`
- `results/04_forecast_metrics.csv`
- `results/04_early_forecast_windows.csv`
- `docs/04_compositional_forecasting.md`

In [ ]:
forecast_path = RESULTS_DIR / "04_compositional_forecasts.csv"
metrics_path = RESULTS_DIR / "04_forecast_metrics.csv"
window_path = RESULTS_DIR / "04_early_forecast_windows.csv"

forecast_df.to_csv(forecast_path, index=False)
metrics.to_csv(metrics_path, index=False)
window_df.to_csv(window_path, index=False)

summary_md = f"""# Notebook 04 — Compositional Forecasting

This notebook explored whether component-task emergence and function-vector geometry can forecast compositional capability emergence.

## Inputs

- emergence order table from Notebook 01
- function-vector similarity/drift outputs from Notebook 02
- optional monitoring flags from Notebook 03

## Forecast heuristic

`predicted checkpoint = max(component checkpoints) + geometry penalty + instability penalty + spread penalty`

## Metrics

{metrics.to_markdown(index=False)}

## Interpretation

Component emergence and function-vector geometry provide a lightweight, interpretable way to estimate compositional emergence.

This does not claim a general predictive law. It provides a monitoring heuristic that can be compared against real checkpoint trajectories.

Emergence ≠ magic. Monitor constraints. 📐
"""

doc_path = DOCS_DIR / "04_compositional_forecasting.md"
doc_path.write_text(summary_md, encoding="utf-8")

print("Saved:")
print("-", forecast_path)
print("-", metrics_path)
print("-", window_path)
print("-", doc_path)

## 12. Interpretation

ElementalTask suggests compositional capabilities emerge in partially stable developmental order.

This notebook explored whether:

- component emergence,
- function-vector geometry,
- and constraint stability

can provide lightweight forecasts of future composite capability emergence during training.

The key contribution is not a black-box predictor. It is a transparent monitoring heuristic for comparing expected and observed compositional emergence.

## 13. Optional: zip figures/results/docs for download

Uncomment this cell in Colab if you want a local zip containing Notebook 04 outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD CELL
# Uncomment to create and download a zip of Notebook 04 outputs.

# import zipfile
# from pathlib import Path

# EXPORT_NAME = "elementaltask_rml_notebook04_outputs.zip"
# export_path = NOTEBOOK_DIR / EXPORT_NAME

# files_to_zip = []
# for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
#     for p in folder.glob("04_*"):
#         if p.is_file():
#             files_to_zip.append(p)

# with zipfile.ZipFile(export_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for p in files_to_zip:
#         zf.write(p, arcname=str(p.relative_to(NOTEBOOK_DIR)))

# print(f"Created: {export_path}")

# try:
#     from google.colab import files
#     files.download(str(export_path))
# except Exception as e:
#     print("Not running in Colab or download unavailable:", e)